# Checks and examples, with and without an ecoinvent licence

**English** | [Français](check_and_examples.fr.ipynb)

This notebook gathers the checks and examples run on the repository:

- **Part 1, with an ecoinvent licence**: import ecoinvent 3.11 cut-off into Brightway, recompute the fabrication impacts and check that they are the same as the shipped ones (computed with Activity Browser) in the 16 EF 3.1 categories; then an example of an inventory change.
- **Part 2, without a licence**: the reference results and three changes to the model.

The expected values are given above each cell. Run the cells in order. In the commands, `!` runs a terminal command and `{sys.executable}` is the Python of the notebook.

## 0. Download the repository and install (once)

Without Git: the repository is downloaded as a ZIP into your home folder. The second `pip` line is only needed for Part 1.

In [ ]:
import sys, os, urllib.request, zipfile
os.chdir(os.path.expanduser("~"))
urllib.request.urlretrieve("https://github.com/Ayabh12/h2-ageing-lca/archive/refs/heads/main.zip", "h2-ageing-lca.zip")
zipfile.ZipFile("h2-ageing-lca.zip").extractall(".")
os.chdir("h2-ageing-lca-main")
!{sys.executable} -m pip install -r requirements.txt
!{sys.executable} -m pip install -U -r requirements-brightway.txt

After installing Brightway, **restart the kernel** (menu *Kernel → Restart*), then go on.

# Part 1 – Check of the fabrication calculation with ecoinvent

Needs an ecoinvent licence (your own or your institution's, if its terms allow it). The login is never written to a file: do not type it in a cell you share.

### 1.1 ecoinvent login

The password is not shown while you type.

In [ ]:
import sys, os
import pandas as pd
os.chdir(os.path.expanduser("~/h2-ageing-lca-main"))
import getpass
os.environ["EI_USERNAME"] = input("ecoinvent username: ")
os.environ["EI_PASSWORD"] = getpass.getpass("ecoinvent password: ")

### 1.2 Import ecoinvent and recompute the fabrication impacts

The first time, ecoinvent 3.11 cut-off is downloaded and imported into the Brightway project `h2ageing` (from about 20 minutes to under an hour). The next times, the calculation takes about one minute.

**Expected**: the list of the 16 EF 3.1 methods used, then the climate change impact of each component: `fuel_cell_stack_128kW` 3698.3, `electrolyser_stack_1p25MW` 39509.4, `tank_type_IV` 2332.0 kg CO2-eq.

In [ ]:
!{sys.executable} scripts/build_fabrication_brightway.py --project h2ageing --import-ecoinvent --out results/fabrication_test.csv

### 1.3 Check of the 16 EF 3.1 categories

Difference (in %) between the impacts recomputed with Brightway and the Activity Browser impacts shipped in the repository.

**Expected**: 0 everywhere (−0 is a rounding); difference below 0.01 % in the 16 categories and for the three components.

In [ ]:
new = pd.read_csv("results/fabrication_test.csv").set_index("category")
shipped = pd.read_csv("data/impacts/fabrication_per_unit.csv").set_index("category")
diff = ((new - shipped) / shipped * 100).round(2)
print("Largest difference:", diff.abs().max().max(), "%")
diff

### 1.4 Example: manufacture the fuel cell stack with Japanese electricity

The Mirai stack is manufactured by Toyota in Japan. The French residual mix is replaced by the Japanese high-voltage electricity market (ecoinvent 3.11) for the 1 096 kWh used to manufacture the stack, then the whole model is run again. The original inventory is restored at the end of the cell.

**Expected**: about 4 316 kg CO2-eq per fuel cell stack instead of 3 698; chain (g CO2-eq/km): no ageing 74.1 → 77.2; central 97.2 → 103.3; rise due to ageing in the central case +31.1 % → +33.8 %.

In [ ]:
p = "data/inventories/fuel_cell_stack_128kW.csv"
original = open(p, encoding="utf-8", newline="").read()
inv = pd.read_csv(p, dtype=str, keep_default_na=False)
row = inv["input"].str.contains("French residual mix")
inv.loc[row, "input"] = "Electricity, Japan, high voltage"
inv.loc[row, "ecoinvent_activity"] = "market for electricity, high voltage"
inv.loc[row, "location"] = "JP"
inv.to_csv(p, index=False)
try:
    !{sys.executable} scripts/build_fabrication_brightway.py --project h2ageing --out results/fabrication_japan.csv
finally:
    open(p, "w", encoding="utf-8", newline="").write(original)   # original inventory restored

!{sys.executable} scripts/run.py --out results/reference > {os.devnull}
!{sys.executable} scripts/run.py --set fabrication.file=results/fabrication_japan.csv --out results/japan > {os.devnull}
ref = pd.read_csv("results/reference/chain.csv").set_index("case")
jp = pd.read_csv("results/japan/chain.csv").set_index("case")
pd.DataFrame({"France: total (g/km)": ref["total_g_per_km"], "Japan: total (g/km)": jp["total_g_per_km"],
              "France: rise (%)": ref["rise_pct"], "Japan: rise (%)": jp["rise_pct"]}).round(1)

# Part 2 – Use and change the model without ecoinvent

The model uses the fabrication impacts shipped in the repository (`data/impacts`). No licence is needed. These results are also checked automatically by `tests/test_reference_results.py`.

In [ ]:
import sys, os
import pandas as pd
os.chdir(os.path.expanduser("~/h2-ageing-lca-main"))

## Example 1 – Reference results

**Expected** (g CO2-eq/km): no ageing 74.1; optimistic 77.8 (+5.0 %); central 97.2 (+31.1 %); pessimistic 100.1 (+35.0 %).

In [ ]:
!{sys.executable} scripts/run.py --out results/reference > {os.devnull}
ref = pd.read_csv("results/reference/chain.csv").set_index("case")
ref[["total_g_per_km", "rise_pct"]].round(1)

## Example 2 – Change the fuel cell lifetime: 6 000 h instead of 4 100 h

**Expected**, central case: 1 fuel cell stack instead of 2; chain 97.2 → 78.9 g/km; rise +31.1 % → +6.4 %. Above 5 554 h, one stack lasts the whole vehicle life.

In [ ]:
!{sys.executable} scripts/run.py --set cases.central.fuel_cell.hours_to_end_of_life=6000 --out results/pile_6000h > {os.devnull}
life = pd.read_csv("results/pile_6000h/life.csv").set_index("case")
p6 = pd.read_csv("results/pile_6000h/chain.csv").set_index("case")
display(life[["fuel_cell_end_of_life_h", "fuel_cell_stacks"]].round(0))
pd.DataFrame({"Reference (g/km)": ref["total_g_per_km"], "Fuel cell 6 000 h (g/km)": p6["total_g_per_km"],
              "Reference rise (%)": ref["rise_pct"], "6 000 h rise (%)": p6["rise_pct"]}).round(1)

## Example 3 – Wind electricity for the electrolyser

**Expected**: no ageing 74.1 → 41.2 g/km; central 97.2 → 61.3 g/km; rise due to ageing +31.1 % → +48.6 %. With clean electricity, the replacement stacks weigh more in proportion. Other choices: `average_mix_FR`, `nuclear_FR`, `photovoltaic`.

In [ ]:
!{sys.executable} scripts/run.py --set usage.electricity_electrolyser=wind_FR --out results/eolien > {os.devnull}
eo = pd.read_csv("results/eolien/chain.csv").set_index("case")
pd.DataFrame({"Residual mix (g/km)": ref["total_g_per_km"], "Wind (g/km)": eo["total_g_per_km"],
              "Residual mix rise (%)": ref["rise_pct"], "Wind rise (%)": eo["rise_pct"]}).round(1)

## Example 4 – Use your own fabrication impact

Test value: fuel cell stack 30 % lower. With a value from another study, replace `*= 0.7` by `= <value>` (kg CO2-eq per stack).

**Expected**: no ageing 74.1 → 68.6 g/km; central 97.2 → 86.1 g/km; rise +31.1 % → +25.5 %.

In [ ]:
fab = pd.read_csv("data/impacts/fabrication_per_unit.csv").set_index("category")
fab.loc["climate_change", "fuel_cell_stack_128kW"] *= 0.7
os.makedirs("results", exist_ok=True)
fab.to_csv("results/my_fabrication.csv")
!{sys.executable} scripts/run.py --set fabrication.file=results/my_fabrication.csv --out results/my_fabrication > {os.devnull}
mf = pd.read_csv("results/my_fabrication/chain.csv").set_index("case")
pd.DataFrame({"Reference (g/km)": ref["total_g_per_km"], "Stack −30 % (g/km)": mf["total_g_per_km"],
              "Reference rise (%)": ref["rise_pct"], "Stack −30 % rise (%)": mf["rise_pct"]}).round(1)

## More examples to test

The next examples change the models themselves: a constant, a law, the model of a case, or a whole case. The function `compare` puts the reference and the variant side by side (whole chain, g CO2-eq/km, and rise due to ageing). Run example 1 first: it computes the reference `ref`.

In [ ]:
import sys
sys.path.insert(0, "src")
from h2ageing import load_config, run

def compare(results, name):
    ch = results["chain"].set_index("case")
    life = results["life"].set_index("case")
    return pd.DataFrame({
        "Reference (g/km)": ref["total_g_per_km"], f"{name} (g/km)": ch["total_g_per_km"],
        "Reference rise (%)": ref["rise_pct"], f"{name} rise (%)": ch["rise_pct"],
        "Fuel cell end of life (h)": life["fuel_cell_end_of_life_h"], "Fuel cell stacks": life["fuel_cell_stacks"],
    }).round(1)

## Example 5 – Change a constant of the semi-empirical model

In the pessimistic case, the fuel cell voltage loss at high current (1.8 A/cm², after 1 000 h) goes from 45 mV to 27 mV, the value published for another cell (see `scripts/semi_empirical_constants.py`).

**Expected**, pessimistic case: fuel cell end of life 2 895 → 5 652 h, 1 stack instead of 2; chain 100.1 → 83.6 g/km; rise +35.0 % → +12.8 %. This is the constant of the semi-empirical model that weighs most.

In [ ]:
cfg = load_config()
cfg["cases"]["pessimistic"]["fuel_cell"]["voltage_loss_at_1p8_A_cm2_1000h_V"] = 0.027
compare(run(cfg), "27 mV")

## Example 6 – Change a law: exponential crossover instead of linear

In the semi-empirical fuel cell model, the hydrogen crossover through the membrane grows linearly by default. Here it grows exponentially (other choice: `"constant"`).

**Expected**, pessimistic case: the end of life does not change (2 895 h, the criterion is on the voltage); over-consumption grows, and the rise goes from +35.0 % to +36.9 % (+32.6 % with a constant crossover). The crossover law weighs little next to the lifetime.

In [ ]:
cfg = load_config()
cfg["fuel_cell"]["crossover_law"] = "exponential"
compare(run(cfg), "exponential")

## Example 7 – Change the model: linear instead of semi-empirical, at the same end of life

In the pessimistic case, the semi-empirical fuel cell model is replaced by the linear model, set to the same end of life (2 895 h). Only the shape of the degradation over time changes.

**Expected**, pessimistic case: 2 stacks in both; chain 100.1 → 99.1 g/km; rise +35.0 % → +33.7 %. At the same end of life, the choice of model changes the result little: the predicted end of life is what matters.

In [ ]:
cfg = load_config()
cfg["cases"]["pessimistic"]["fuel_cell"] = {"model": "linear", "hours_to_end_of_life": 2895}
compare(run(cfg), "linear 2 895 h")

## Example 8 – Create your own ageing case

A fourth case is added, with its own values: fuel cell of 6 000 h, electrolyser at 3.0 µV/h per cell, tank permeation from 6 to 10 mL/h/L. These are test values: replace them with yours.

**Expected**, case `my_case`: 1 fuel cell stack, 2 electrolyser stacks; chain 78.3 g/km; rise +5.6 %.

In [ ]:
cfg = load_config()
cfg["cases"]["my_case"] = {
    "fuel_cell": {"model": "linear", "hours_to_end_of_life": 6000},
    "electrolyser": {"model": "linear", "rate_uV_per_h_per_cell": 3.0},
    "tank_permeation_mL_h_L": [6.0, 10.0],
}
res = run(cfg)
pd.concat([res["chain"].set_index("case")[["total_g_per_km", "rise_pct"]],
           res["life"].set_index("case")[["fuel_cell_stacks", "electrolyser_stacks"]]], axis=1).round(1)

## Example 9 – Change the mileage: 150 000 km instead of 200 000 km

**Expected**: fabrication is spread over fewer kilometres, so the impact per km rises (no ageing 74.1 → 84.2 g/km; central 97.2 → 113.9 g/km); the central rise goes from +31.1 % to +35.2 %, because 2 fuel cell stacks are still needed.

In [ ]:
cfg = load_config()
cfg["horizon"]["vehicle_km_total"] = 150000
compare(run(cfg), "150 000 km")

# Part 3 – All impact categories and sensitivity

Without ecoinvent. Run example 1 of Part 2 first: it writes the reference results in `results/reference`.

In [ ]:
import sys, os
import pandas as pd
import matplotlib.pyplot as plt
os.chdir(os.path.expanduser("~/h2-ageing-lca-main"))
sys.path.insert(0, "src")
from h2ageing import load_config, run

## 3.1 The 16 EF 3.1 categories

Rise due to ageing of the whole chain in each category, for the three cases, and share of fabrication in the fuel cell impact (case without ageing).

**Expected**, central case: from +9.9 % (ionising radiation) to +93.6 % (ozone depletion); climate change +31.1 %, acidification +86.9 %, mineral and metal resources +72.3 %. The rise is largest in the categories where stack fabrication weighs most (platinum, steel, PTFE). EF 3.1 single score: +5.8 %, +29.2 % and +34.0 %.

In [ ]:
cat = pd.read_csv("results/reference/categories.csv").set_index("category")
tab = cat[["unit", "chain_rise_pct_optimistic", "chain_rise_pct_central", "chain_rise_pct_pessimistic",
           "fuel_cell_fabrication_share_pct"]].round(1)
tab.columns = ["Unit", "Optimistic rise (%)", "Central rise (%)", "Pessimistic rise (%)",
               "Fabrication share, fuel cell (%)"]
score = pd.read_csv("results/reference/chain.csv").set_index("case")["single_score_rise_pct"]
print("EF 3.1 single score, rise (%):", score.drop("no_ageing").round(1).to_dict())
tab.sort_values("Central rise (%)", ascending=False)

In [ ]:
t = tab.sort_values("Central rise (%)")
ax = t["Central rise (%)"].plot.barh(figsize=(7, 6), color="#3b6ea5")
ax.set_xlabel("Rise due to ageing, central case (%)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 3.2 Sensitivity to the electricity source

The electrolyser is supplied in turn by the five sources shipped.

**Expected** (chain, g CO2-eq/km; no ageing → central; central rise): residual mix 74.1 → 97.2 (+31.1 %); average mix 73.0 → 96.0 (+31.4 %); nuclear 36.1 → 55.6 (+54.3 %); wind 41.2 → 61.3 (+48.6 %); photovoltaic 50.7 → 71.6 (+41.3 %). The cleaner the electricity, the more ageing weighs in proportion.

In [ ]:
rows = []
for source in ["residual_mix_FR", "average_mix_FR", "nuclear_FR", "wind_FR", "photovoltaic"]:
    cfg = load_config()
    cfg["usage"]["electricity_electrolyser"] = source
    ch = run(cfg)["chain"].set_index("case")
    rows.append({"Electricity": source,
                   "No ageing (g/km)": ch.loc["no_ageing", "total_g_per_km"],
                   "Central (g/km)": ch.loc["central", "total_g_per_km"],
                   "Central rise (%)": ch.loc["central", "rise_pct"]})
pd.DataFrame(rows).round(1)

## 3.3 Fuel cell lifetime threshold

The fuel cell stack lifetime varies from 1 500 h to 12 000 h (linear model); the three cases are placed on the curve. About 30 seconds.

**Expected**: the rise of the chain falls in steps. Above 5 554 h (the fuel cell operating hours over 200 000 km), one stack is enough and the rise drops to about +6 %.

In [ ]:
!{sys.executable} scripts/threshold.py --out results/threshold > {os.devnull}
threshold = pd.read_csv("results/threshold/threshold_fuel_cell.csv")
ax = threshold.plot(x="lifetime_h", y="chain_rise_pct", legend=False, figsize=(7, 4), color="#3b6ea5")
for h, nom in [(2895, "pessimistic"), (4100, "central"), (8000, "optimistic")]:
    ax.axvline(h, color="grey", ls=":")
    ax.text(h, ax.get_ylim()[1] * 0.95, nom, rotation=90, va="top", ha="right")
ax.axvline(5554, color="firebrick", ls="--")
ax.set_xlabel("Fuel cell stack lifetime (h)")
ax.set_ylabel("Rise of the chain (%)")
plt.tight_layout()
plt.show()

## 3.4 Global sensitivity analysis (Sobol)

The eight uncertain inputs vary together, within the ranges of `global_sensitivity` in `config/default.yaml`. The total Sobol index (ST) gives the share of the variation of the result due to each input, alone or with the others. To be quick, this cell uses 128 samples (1 280 runs, about 2 minutes); the reference results use 1 024 samples (`--samples 1024`, about 15 minutes).

**Expected** (reference values, 1 024 samples): for the rise due to ageing, fuel cell lifetime ST ≈ 0.94 and electricity source ≈ 0.09; for the absolute impact, electricity source ≈ 0.78 and fuel cell lifetime ≈ 0.18. With 128 samples, the values are close, with wider intervals.

In [ ]:
!{sys.executable} scripts/sobol.py --samples 128 --out results/sobol > {os.devnull}
sob = pd.read_csv("results/sobol/sobol_indices.csv")
st = sob.pivot(index="factor", columns="output", values="ST").round(2)
st.columns = ["Absolute impact (ST)", "Rise due to ageing (ST)"]
st.sort_values("Rise due to ageing (ST)", ascending=False)